# 01 — Exploratory Data Analysis
**EAWeather BI Pipeline** | Nairobi · Mombasa · Kampala · Dar es Salaam

This notebook loads data straight from PostgreSQL and runs a full EDA:
shape checks, null counts, temperature trends, AQI distributions, and correlations.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

CITY_COLORS = {
    "Nairobi":       "#006B6B",
    "Mombasa":       "#E87722",
    "Kampala":       "#4A90D9",
    "Dar es Salaam": "#7B68EE",
}

In [ ]:
# ── DB connection ──────────────────────────────────────────────────────────
engine = create_engine(
    f"postgresql+psycopg2://"
    f"{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', 5432)}"
    f"/{os.getenv('DB_NAME', 'eaweather')}"
)

# ── Load tables ────────────────────────────────────────────────────────────
with engine.connect() as conn:
    weather = pd.read_sql(
        "SELECT c.city_name, c.country, w.* "
        "FROM weather_readings w JOIN cities c USING (city_id) "
        "ORDER BY w.recorded_at",
        conn
    )
    aqi = pd.read_sql(
        "SELECT c.city_name, c.country, a.* "
        "FROM air_quality_readings a JOIN cities c USING (city_id) "
        "ORDER BY a.recorded_at",
        conn
    )
    summaries = pd.read_sql(
        "SELECT c.city_name, s.* "
        "FROM daily_summaries s JOIN cities c USING (city_id) "
        "ORDER BY s.summary_date",
        conn
    )

weather["recorded_at"]  = pd.to_datetime(weather["recorded_at"],  utc=True)
aqi["recorded_at"]       = pd.to_datetime(aqi["recorded_at"],       utc=True)
summaries["summary_date"] = pd.to_datetime(summaries["summary_date"])

print(f"Weather readings : {len(weather):,} rows across {weather['city_name'].nunique()} cities")
print(f"AQI readings     : {len(aqi):,} rows across {aqi['city_name'].nunique()} cities")
print(f"Daily summaries  : {len(summaries):,} rows")

## 1 · Schema & Null Audit

In [ ]:
print("=== WEATHER ===")
print(weather.dtypes, "\n")
print(weather.isnull().sum().rename("nulls"))

In [ ]:
print("=== AIR QUALITY ===")
print(aqi.dtypes, "\n")
print(aqi.isnull().sum().rename("nulls"))

## 2 · Descriptive Statistics

In [ ]:
weather_cols = ["temperature_c", "feels_like_c", "humidity_pct",
                "pressure_hpa", "wind_speed_ms", "cloudiness_pct"]
weather.groupby("city_name")[weather_cols].describe().round(2)

In [ ]:
aqi_cols = ["aqi", "pm2_5", "pm10", "no2", "o3", "co"]
aqi.groupby("city_name")[aqi_cols].describe().round(2)

## 3 · Temperature Trends

In [ ]:
fig, ax = plt.subplots()
for city, grp in weather.groupby("city_name"):
    ax.plot(grp["recorded_at"], grp["temperature_c"],
            label=city, color=CITY_COLORS.get(city), alpha=0.8, linewidth=1.2)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.set(title="Temperature Over Time (°C)", xlabel="", ylabel="Temperature (°C)")
ax.legend()
plt.tight_layout()
plt.show()

## 4 · AQI Distribution by City

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
aqi.boxplot(column="aqi", by="city_name", ax=axes[0],
            patch_artist=True,
            boxprops=dict(facecolor="#E0F2F1"))
axes[0].set(title="AQI Distribution by City", xlabel="", ylabel="AQI (1–5)")
axes[0].get_figure().suptitle("")

# Bar: mean AQI
mean_aqi = aqi.groupby("city_name")["aqi"].mean().sort_values()
mean_aqi.plot.barh(ax=axes[1], color=[CITY_COLORS.get(c, "#aaa") for c in mean_aqi.index])
axes[1].set(title="Mean AQI by City", xlabel="Mean AQI", ylabel="")
for bar, val in zip(axes[1].patches, mean_aqi):
    axes[1].text(val + 0.02, bar.get_y() + bar.get_height()/2,
                 f"{val:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 5 · AQI Time Series

In [ ]:
fig, ax = plt.subplots()
for city, grp in aqi.groupby("city_name"):
    ax.plot(grp["recorded_at"], grp["aqi"],
            label=city, color=CITY_COLORS.get(city), alpha=0.85, linewidth=1.2)

# Shade AQI zones
ax.axhspan(1, 2, alpha=0.05, color="green",  label="_Good")
ax.axhspan(2, 3, alpha=0.05, color="yellow", label="_Fair")
ax.axhspan(3, 4, alpha=0.05, color="orange", label="_Moderate")
ax.axhspan(4, 5, alpha=0.05, color="red",    label="_Poor/Very Poor")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.set(title="AQI Over Time (1=Good → 5=Very Poor)",
       xlabel="", ylabel="AQI", ylim=(0.8, 5.5))
ax.legend()
plt.tight_layout()
plt.show()

## 6 · Correlation Heatmap

In [ ]:
# Merge weather + AQI on nearest timestamp per city
merged = pd.merge_asof(
    weather.sort_values("recorded_at"),
    aqi[["city_name","recorded_at","aqi","pm2_5","pm10","no2","o3"]].sort_values("recorded_at"),
    on="recorded_at", by="city_name", tolerance=pd.Timedelta("2h"),
    suffixes=("", "_aqi")
).dropna(subset=["aqi"])

corr_cols = ["temperature_c", "humidity_pct", "pressure_hpa",
             "wind_speed_ms", "cloudiness_pct", "aqi", "pm2_5", "pm10"]
corr = merged[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 7 · Weather Description Frequency

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, (city, grp) in enumerate(weather.groupby("city_name")):
    counts = grp["weather_main"].value_counts().head(6)
    counts.plot.bar(ax=axes[i], color=CITY_COLORS.get(city, "#888"), edgecolor="white")
    axes[i].set(title=city, xlabel="", ylabel="Readings")
    axes[i].tick_params(axis="x", rotation=30)

plt.suptitle("Weather Condition Frequency per City", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 8 · Top 10 Worst AQI Readings

In [ ]:
worst = (
    aqi[["city_name", "recorded_at", "aqi", "aqi_category", "pm2_5", "pm10"]]
    .sort_values("aqi", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
worst["recorded_at"] = worst["recorded_at"].dt.strftime("%Y-%m-%d %H:%M UTC")
worst.index += 1
worst